# PHASE 1: Data Acquisition & Environment Setup
**Traceability**
- Dataset DOI: [10.1109/PHM.2008.4711411](https://doi.org/10.1109/PHM.2008.4711411)
- Issue ID: #1 Data Acquisition & Environment Setup

## 1. Objectives
- Establish a reproducible environment for NASA CMAPSS analysis.
- Load the NASA CMAPSS FD001 dataset (train, test, and RUL ground truth).
- Validate the schema to ensure data quality and prevent downstream errors.

### 1.1 Import Libraries & Configure Environment
We start by importing essential libraries and setting a global random seed to ensure reproducibility.

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# ── Reproducibility Config ──────────────────────────────────────────────
np.random.seed(42)

# ── Global Config ────────────────────────────────────────────────────────
DATA_DIR = Path('../data')
COL_NAMES = (
    ['unit_number', 'time_cycles'] +
    ['setting_1', 'setting_2', 'setting_3'] +
    [f's_{i}' for i in range(1, 22)]   # s_1 ... s_21
)

### 1.2 Data Loading Methods
Define a helper function to load the whitespace-separated text files into pandas DataFrames.

In [2]:
def load_dataset(filename):
    """Load whitespace-separated text file into a DataFrame."""
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {path}")
    
    df = pd.read_csv(
        path,
        sep=r'\s+', header=None, index_col=False, names=COL_NAMES
    )
    return df

### 1.3 Schema Validation Methods
Define a function to perform various data quality checks, including shape, null values, and engine start cycles.

In [3]:
def validate_schema(df, name):
    """Perform schema validation checks."""
    print(f"\n{'='*50}")
    print(f"  SCHEMA VALIDATION — {name}")
    print(f"{'='*50}")
    
    # 1. Shape Check
    print(f"  Shape          : {df.shape}")
    
    # 2. Dtype Check
    is_numeric = all(df.dtypes != 'object')
    print(f"  Dtypes OK (All Numeric): {is_numeric}")
    
    # 3. Null Count
    null_count = df.isnull().sum().sum()
    print(f"  Null count     : {null_count}")
    
    # 4. Duplicates
    dup_count = df.duplicated().sum()
    print(f"  Duplicates     : {dup_count}")
    
    # 5. Engine Consistency
    n_engines = df['unit_number'].nunique()
    print(f"  unit_number    : {n_engines} unique engines")
    
    starts_at_1 = (df.groupby('unit_number')['time_cycles'].min() == 1).all()
    print(f"  All engines start at cycle 1: {starts_at_1}")
    
    return {
        "shape": df.shape,
        "is_numeric": is_numeric,
        "null_count": null_count,
        "dup_count": dup_count,
        "n_engines": n_engines,
        "starts_at_1": starts_at_1
    }

### 1.4 Execution: Load & Validate
Finally, we load the training and test sets along with the ground truth RUL, and run our validation checks.

In [4]:
# 1. Load All Files
df_train = load_dataset('train_FD001.txt')
df_test = load_dataset('test_FD001.txt')

# Load RUL ground truth
rul_path = DATA_DIR / 'RUL_FD001.txt'
y_test_rul = pd.read_csv(
    rul_path,
    sep=r'\s+', header=None, index_col=False, names=['RUL']
)

print(f"✅ Data loaded successfully.")

# 2. Validate Schema
val_train = validate_schema(df_train, "TRAIN FD001")
val_test = validate_schema(df_test, "TEST FD001")

print(f"\n  RUL ground truth shape: {y_test_rul.shape}")

✅ Data loaded successfully.

  SCHEMA VALIDATION — TRAIN FD001
  Shape          : (20631, 26)
  Dtypes OK (All Numeric): True
  Null count     : 0
  Duplicates     : 0
  unit_number    : 100 unique engines
  All engines start at cycle 1: True

  SCHEMA VALIDATION — TEST FD001
  Shape          : (13096, 26)
  Dtypes OK (All Numeric): True
  Null count     : 0
  Duplicates     : 0
  unit_number    : 100 unique engines
  All engines start at cycle 1: True

  RUL ground truth shape: (100, 1)
